In [ ]:
from importlib import reload

import os
from pathlib import Path
# Set the project folder
PRJ_PATH = Path("/Users/admin/Desktop/Monica/PhD")
INC_PATH = os.path.join(PRJ_PATH, "inc")
IMG_PATH = os.path.join(PRJ_PATH, "img")
DATA_PATH = os.path.join(PRJ_PATH, "data")
import sys
# Add to path
if INC_PATH not in sys.path: sys.path.append(INC_PATH)
import argparse
import pandas as pd
from obspy import UTCDateTime
from copy import deepcopy as dcpy

from constants import *
import analyzer as ana
import initializer as ini

args = {
  'channel': None,
  'file': [Path('/Users/admin/Desktop/Monica/PhD/data/manual')],
  'groups': [DATE_STR, NETWORK_STR, STATION_STR],
  'key': None,
  'models': [PHASENET_STR, EQTRANSFORMER_STR],
  'network': [ALL_WILDCHAR_STR],
  'station': [ALL_WILDCHAR_STR],
  'train': False,
  'weights': [INSTANCE_STR, ORIGINAL_STR, STEAD_STR, SCEDC_STR],
  'batch': 4096,
  'config': None,
  'directory': Path('/Users/admin/Desktop/Monica/PhD/data/waveforms'),
  'option': '*',
  'pwave': 0.1,
  'swave': 0.1,
  'client': ['http://158.110.30.217:8080'],
  'denoiser': False,
  'download': False,
  'interactive': False,
  'force': False,
  'pyrocko': False,
  'pyocto': False,
  'timing': False,
  'dates': [UTCDateTime(2023, 6, 1, 0, 0), UTCDateTime(2023, 12, 31, 0, 0)],
  'julian': None,
  'rectdomain': [44, 47.5, 9.5, 15], # [10, 14.5, 44.5, 47]
  'circdomain': None,
  'silent': False,
  'verbose': True
}
args = argparse.Namespace(**args)
DATA_PATH = args.directory.parent

In [ ]:
reload(ini)
WAVEFORMS = ini.waveform_table(args)

In [ ]:
reload(ini)
INVENTORY, STATIONS = ini.station_loader(args, WAVEFORMS)

In [ ]:
reload(ini)
TRUE_S, TRUE_D = ini.true_loader(args, WAVEFORMS, INVENTORY, STATIONS)

In [ ]:
reload(ana)
PRED = ana._Analysis(args, CLSSFD_STR)
PRED_TP = ana.stat_test(dcpy(TRUE_D), dcpy(PRED), args, CLSSFD_STR)

In [ ]:
reload(ana)
ana.time_displacement(dcpy(PRED_TP), args, CLSSFD_STR)

In [ ]:
reload(ana)
print(SOURCE_STR)
PRED_S = ini.data_loader(Path(
  DATA_PATH, ("D_" if args.denoiser else EMPTY_STR) + SOURCE_STR + 
  CSV_EXT))
PRED_S[TIMESTAMP_STR] = PRED_S[TIMESTAMP_STR].apply(lambda x:
                                                    UTCDateTime(x))

In [ ]:
reload(ana)
ana.time_displacement(ana.stat_test(TRUE_S, PRED_S, args, SOURCE_STR), args,
                      SOURCE_STR)